### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="taiwanese_bankruptcy_prediction",
    dataset_year="2009",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5004D",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/taiwanese_bankruptcy_prediction/ && wget -P local-data-warehouse/taiwanese_bankruptcy_prediction/ https://archive.ics.uci.edu/static/public/572/taiwanese+bankruptcy+prediction.zip && unzip local-data-warehouse/taiwanese_bankruptcy_prediction/taiwanese+bankruptcy+prediction.zip -d local-data-warehouse/taiwanese_bankruptcy_prediction/ && rm local-data-warehouse/taiwanese_bankruptcy_prediction/taiwanese+bankruptcy+prediction.zip
""",
    # References
    academic_reference_bibtex="""@article{liang2016financial,
  title={Financial ratios and corporate governance indicators in bankruptcy prediction: A comprehensive study},
  author={Liang, Deron and Lu, Chia-Chi and Tsai, Chih-Fong and Shih, Guan-An},
  journal={European journal of operational research},
  volume={252},
  number={2},
  pages={561--572},
  year={2016},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="liang2016financial",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We map binary Bankrupt column to yes/no
- We rename the features to be shorter and have no special characters.
- We drop the constant column "Net_Income_Flag".
- We drop two duplicated features (same values as other features).
- Anomaly: the paper introducing the dataset has more features, but these do not seem to be in the dataset. 
- Anomaly: the data is temporal but all features are time-invariant. This might still have a latent, non-verifiable temporal component, which we ignore for our predictive IID task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Bankrupt",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Bankrupt",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/data.csv")

df.columns = [
    "Bankrupt",
    "ROA_C_Before_Interest_Depreciation",
    "ROA_A_Before_Interest_After_Tax",
    "ROA_B_Before_Interest_Depreciation_After_Tax",
    "Operating_Gross_Margin",
    "Realized_Sales_Gross_Margin",
    "Operating_Profit_Rate",
    "PreTax_Net_Interest_Rate",
    "AfterTax_Net_Interest_Rate",
    "NonIndustry_Income_Expenditure_Revenue",
    "Continuous_Interest_Rate_After_Tax",
    "Operating_Expense_Rate",
    "R&D_Expense_Rate",
    "Cash_Flow_Rate",
    "InterestBearing_Debt_Interest_Rate",
    "Tax_Rate_A",
    "Net_Value_Per_Share_B",
    "Net_Value_Per_Share_A",
    "Net_Value_Per_Share_C",
    "Persistent_EPS_Last_4_Seasons",
    "Cash_Flow_Per_Share",
    "Revenue_Per_Share",
    "Operating_Profit_Per_Share",
    "Net_Profit_Before_Tax_Per_Share",
    "Realized_Sales_Gross_Profit_Growth_Rate",
    "Operating_Profit_Growth_Rate",
    "AfterTax_Net_Profit_Growth_Rate",
    "Regular_Net_Profit_Growth_Rate",
    "Continuous_Net_Profit_Growth_Rate",
    "Total_Asset_Growth_Rate",
    "Net_Value_Growth_Rate",
    "Total_Asset_Return_Growth_Rate",
    "Cash_Reinvestment_Percent",
    "Current_Ratio",
    "Quick_Ratio",
    "Interest_Expense_Ratio",
    "Total_Debt_to_Net_Worth",
    "Debt_Ratio_Percent",
    "Net_Worth_to_Assets",
    "LongTerm_Fund_Suitability_Ratio_A",
    "Borrowing_Dependency",
    "Contingent_Liabilities_to_Net_Worth",
    "Operating_Profit_to_PaidIn_Capital",
    "Net_Profit_Before_Tax_to_PaidIn_Capital",
    "Inventory_Accounts_Receivable_to_Net_Value",
    "Total_Asset_Turnover",
    "Accounts_Receivable_Turnover",
    "Average_Collection_Days",
    "Inventory_Turnover_Rate",
    "Fixed_Assets_Turnover_Frequency",
    "Net_Worth_Turnover_Rate",
    "Revenue_Per_Person",
    "Operating_Profit_Per_Person",
    "Allocation_Rate_Per_Person",
    "Working_Capital_to_Total_Assets",
    "Quick_Assets_to_Total_Assets",
    "Current_Assets_to_Total_Assets",
    "Cash_to_Total_Assets",
    "Quick_Assets_to_Current_Liability",
    "Cash_to_Current_Liability",
    "Current_Liability_to_Assets",
    "Operating_Funds_to_Liability",
    "Inventory_to_Working_Capital",
    "Inventory_to_Current_Liability",
    "Current_Liabilities_to_Liability",
    "Working_Capital_to_Equity",
    "Current_Liabilities_to_Equity",
    "LongTerm_Liability_to_Current_Assets",
    "Retained_Earnings_to_Total_Assets",
    "Total_Income_to_Total_Expense",
    "Total_Expense_to_Assets",
    "Current_Asset_Turnover_Rate",
    "Quick_Asset_Turnover_Rate",
    "Working_Capital_Turnover_Rate",
    "Cash_Turnover_Rate",
    "Cash_Flow_to_Sales",
    "Fixed_Assets_to_Assets",
    "Current_Liability_to_Liability",
    "Current_Liability_to_Equity",
    "Equity_to_LongTerm_Liability",
    "Cash_Flow_to_Total_Assets",
    "Cash_Flow_to_Liability",
    "CFO_to_Assets",
    "Cash_Flow_to_Equity",
    "Current_Liability_to_Current_Assets",
    "Liability_Assets_Flag",
    "Net_Income_to_Total_Assets",
    "Total_Assets_to_GNP_Price",
    "NoCredit_Interval",
    "Gross_Profit_to_Sales",
    "Net_Income_to_Stockholders_Equity",
    "Liability_to_Equity",
    "DFL",
    "Interest_Coverage_Ratio",
    "Net_Income_Flag",
    "Equity_to_Liability",
]

cat_features = [
    "Bankrupt",
]
df["Bankrupt"] = df["Bankrupt"].map({0: "No", 1: "Yes"})
df[cat_features] = df[cat_features].astype("category")

df = df.drop(columns=[
    "Net_Income_Flag",
    "Current_Liability_to_Liability",
    "Current_Liability_to_Equity",
])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,819
Columns: 93
Use sampling: False (sample size: 6,819)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Working_Capital_to_Equity', 'Working_Capital_Turnover_Rate', 'Cash_Flow_to_Sales', 'Total_Expense_to_Assets', 'Total_Income_to_Total_Expense', 'Operating_Funds_to_Liability', 'Retained_Earnings_to_Total_Assets', 'Current_Liability_to_Current_Assets', 'Net_Income_to_Total_Assets', 'Cash_Flow_to_Equity']
Rows remaining as candidates after top-10 filter: 0 (of 6,819)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Bankrupt,ROA_C_Before_Interest_Depreciation,ROA_A_Before_Interest_After_Tax,ROA_B_Before_Interest_Depreciation_After_Tax,Operating_Gross_Margin,Realized_Sales_Gross_Margin,Operating_Profit_Rate,PreTax_Net_Interest_Rate,AfterTax_Net_Interest_Rate,NonIndustry_Income_Expenditure_Revenue,Continuous_Interest_Rate_After_Tax,Operating_Expense_Rate,R&D_Expense_Rate,Cash_Flow_Rate,InterestBearing_Debt_Interest_Rate,Tax_Rate_A,Net_Value_Per_Share_B,Net_Value_Per_Share_A,Net_Value_Per_Share_C,Persistent_EPS_Last_4_Seasons,Cash_Flow_Per_Share,Revenue_Per_Share,Operating_Profit_Per_Share,Net_Profit_Before_Tax_Per_Share,Realized_Sales_Gross_Profit_Growth_Rate,Operating_Profit_Growth_Rate,AfterTax_Net_Profit_Growth_Rate,Regular_Net_Profit_Growth_Rate,Continuous_Net_Profit_Growth_Rate,Total_Asset_Growth_Rate,Net_Value_Growth_Rate,Total_Asset_Return_Growth_Rate,Cash_Reinvestment_Percent,Current_Ratio,Quick_Ratio,Interest_Expense_Ratio,Total_Debt_to_Net_Worth,Debt_Ratio_Percent,Net_Worth_to_Assets,LongTerm_Fund_Suitability_Ratio_A,Borrowing_Dependency,Contingent_Liabilities_to_Net_Worth,Operating_Profit_to_PaidIn_Capital,Net_Profit_Before_Tax_to_PaidIn_Capital,Inventory_Accounts_Receivable_to_Net_Value,Total_Asset_Turnover,Accounts_Receivable_Turnover,Average_Collection_Days,Inventory_Turnover_Rate,Fixed_Assets_Turnover_Frequency,Net_Worth_Turnover_Rate,Revenue_Per_Person,Operating_Profit_Per_Person,Allocation_Rate_Per_Person,Working_Capital_to_Total_Assets,Quick_Assets_to_Total_Assets,Current_Assets_to_Total_Assets,Cash_to_Total_Assets,Quick_Assets_to_Current_Liability,Cash_to_Current_Liability,Current_Liability_to_Assets,Operating_Funds_to_Liability,Inventory_to_Working_Capital,Inventory_to_Current_Liability,Current_Liabilities_to_Liability,Working_Capital_to_Equity,Current_Liabilities_to_Equity,LongTerm_Liability_to_Current_Assets,Retained_Earnings_to_Total_Assets,Total_Income_to_Total_Expense,Total_Expense_to_Assets,Current_Asset_Turnover_Rate,Quick_Asset_Turnover_Rate,Working_Capital_Turnover_Rate,Cash_Turnover_Rate,Cash_Flow_to_Sales,Fixed_Assets_to_Assets,Equity_to_LongTerm_Liability,Cash_Flow_to_Total_Assets,Cash_Flow_to_Liability,CFO_to_Assets,Cash_Flow_to_Equity,Current_Liability_to_Current_Assets,Liability_Assets_Flag,Net_Income_to_Total_Assets,Total_Assets_to_GNP_Price,NoCredit_Interval,Gross_Profit_to_Sales,Net_Income_to_Stockholders_Equity,Liability_to_Equity,DFL,Interest_Coverage_Ratio,Equity_to_Liability
0,No,0.434456,0.481247,0.498742,0.596326,0.596326,0.998791,0.797012,0.809041,0.303237,0.781291,2.217908e-04,3.040000e+09,0.473117,0.000000,0.000000,0.174076,0.174076,0.174076,0.202515,0.320153,0.009755,0.087452,0.156963,0.021977,0.842429,0.685817,0.685817,0.217409,5.620000e+09,0.000419,0.263370,0.379056,0.016739,0.014061,0.630611,0.001595,0.043807,0.956193,0.005102,0.369637,0.007023,0.087466,0.155991,0.395832,0.065967,0.000781,0.008093,2.967364e-04,5.900000e+09,0.017419,0.004304,0.389498,0.009512,0.787113,0.185681,0.204108,0.046863,0.014760,0.010498,0.023262,0.357598,0.277073,0.005461,0.472822,0.733696,0.326899,0.000000,0.922162,0.002054,0.029762,1.104112e-04,1.054056e-04,0.593969,7.230000e+09,0.671572,0.624778,0.110933,0.642917,0.459268,0.590226,0.314565,0.017526,0,0.765336,0.001373,0.626305,0.596326,0.838369,0.275936,0.026791,0.565157,0.087378
1,No,0.542534,0.571413,0.590663,0.603417,0.603417,0.999041,0.797476,0.809375,0.303526,0.781638,8.550000e+09,1.550000e+09,0.467422,0.000264,0.251645,0.189078,0.189078,0.189078,0.232107,0.329561,0.053753,0.115300,0.188522,0.022079,0.848008,0.689377,0.689377,0.217591,6.220000e+09,0.000466,0.263868,0.385297,0.009777,0.005425,0.631052,0.007164,0.130000,0.870000,0.005211,0.376524,0.005366,0.115280,0.187588,0.404142,0.193403,0.001051,0.006012,1.282100e-04,2.632612e-04,0.048387,0.018736,0.396938,0.008633,0.809259,0.384289,0.556516,0.049034,0.006546,0.002434,0.106642,0.354767,0.277414,0.008694,0.775494,0.736243,0.331198,0.004023,0.946527,0.002402,0.021704,1.051830e-04,7.420000e+09,0.593946,2.720

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Bankrupt,category,0.0,0.0,2.0,"No, Yes"
1,ROA_C_Before_Interest_Depreciation,float64,0.0,0.0,3333.0,"0.4901, 0.5165, 0.4992, 0.5138, 0.5019, 0.4804, 0.5146, 0.4841, 0.4951, 0.5061"
2,ROA_A_Before_Interest_After_Tax,float64,0.0,0.0,3151.0,"0.5597, 0.5683, 0.5579, 0.5631, 0.566, 0.5589, 0.5542, 0.5613, 0.5806, 0.5511"
3,ROA_B_Before_Interest_Depreciation_After_Tax,float64,0.0,0.0,3160.0,"0.5515, 0.5525, 0.5582, 0.5388, 0.5523, 0.554, 0.5405, 0.5436, 0.5434, 0.5507"
4,Operating_Gross_Margin,float64,0.0,0.0,3781.0,"0.599, 0.6065, 0.602, 0.6058, 0.6009, 0.6083, 0.6044, 0.5992, 0.6052, 0.5982"
5,Realized_Sales_Gross_Margin,float64,0.0,0.0,3788.0,"0.6026, 0.6058, 0.6047, 0.6003, 0.6027, 0.607, 0.6065, 0.599, 0.607, 0.5982"
6,Operating_Profit_Rate,float64,0.0,0.0,3376.0,"0.999, 0.999, 0.999, 0.999, 0.999, 0.999, 0.999, 0.999, 0.999, 0.999"
7,PreTax_Net_Interest_Rate,float64,0.0,0.0,3789.0,"0.7974, 0.7974, 0.7974, 0.7974, 0.7974, 0.7974, 0.7975, 0.7974, 0.7975, 0.7975"
8,AfterTax_Net_Interest_Rate,float64,0.0,0.0,3604.0,"0.8093, 0.8094, 0.8093, 0.8093, 0.8093, 0.8093, 0.8093, 0.8093, 0.8093, 0.8094"
9,NonIndustry_Income_Expenditure_Revenue,float64,0.0,0.0,2551.0,"0.3035, 0.3035, 0.3035, 0.3035, 0.3035, 0.3035, 0.3035, 0.3035, 0.3035, 0.3035"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
ROA_C_Before_Interest_Depreciation,6819.0,5.051796e-01,6.068564e-02,0.0,1.000000e+00
ROA_A_Before_Interest_After_Tax,6819.0,5.586249e-01,6.562003e-02,0.0,1.000000e+00
ROA_B_Before_Interest_Depreciation_After_Tax,6819.0,5.535887e-01,6.159481e-02,0.0,1.000000e+00
Operating_Gross_Margin,6819.0,6.079480e-01,1.693381e-02,0.0,1.000000e+00
Realized_Sales_Gross_Margin,6819.0,6.079295e-01,1.691607e-02,0.0,1.000000e+00
Operating_Profit_Rate,6819.0,9.987551e-01,1.301003e-02,0.0,1.000000e+00
PreTax_Net_Interest_Rate,6819.0,7.971898e-01,1.286899e-02,0.0,1.000000e+00
AfterTax_Net_Interest_Rate,6819.0,8.090836e-01,1.360065e-02,0.0,1.000000e+00
NonIndustry_Income_Expenditure_Revenue,6819.0,3.036229e-01,1.116344e-02,0.0,1.000000e+00
Continuous_Interest_Rate_After_Tax,6819.0,7.813814e-01,1.267900e-02,0.0,1.000000e+00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                    
Bankrupt 1       No   6599  96.77
         2      Yes    220   3.23

In [8]:
# Target Distribution
target_df

,count,pct
Bankrupt,,
No,6599,96.77
Yes,220,3.23


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to taiwanese_bankruptcy_prediction/019d5d97-a099-74fa-a2ca-48f596397dfa
019d5d97-a099-74fa-a2ca-48f596397dfa
4677ce7616fa453a4116ebdd9832515808185def935cebb8eec64edb6e31628f
